# Himawari-8 ARP and CLP Datetime Processing

Process both consolidated products in memory. The original CSV files are not modified.

In [26]:
from pathlib import Path
from datetime import time

import pandas as pd

DATA_DIR = Path("dataset/row/himawari8")
PRODUCTS = ("CLP", "ARP")
YEAR_GROUPS = ("2016_2020", "2026")
PROJECT_ROOT = next(
    (
        root
        for root in (Path.cwd(), *Path.cwd().parents)
        if all(
            (root / DATA_DIR / product / f"{product}_{year_group}.csv").is_file()
            for product in PRODUCTS
            for year_group in YEAR_GROUPS
        )
    ),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        f"Could not locate consolidated ARP and CLP CSV files from {Path.cwd()}"
    )

datasets = {
    product: [
        pd.read_csv(
            PROJECT_ROOT / DATA_DIR / product / f"{product}_{year_group}.csv",
        )
        for year_group in YEAR_GROUPS
    ]
    for product in PRODUCTS
}

print(f"Project root: {PROJECT_ROOT}")
for product, product_datasets in datasets.items():
    print(f"Loaded {product}: {len(product_datasets)} datasets")

Project root: c:\FYP\coding
Loaded CLP: 2 datasets
Loaded ARP: 2 datasets


## Task 1: Datetime Processing

The Himawari-8 generation code defines `timestamp` as UTC and derives `malaysia_datetime` by adding eight hours. For both ARP and CLP, verify that the existing Malaysia-local value agrees with converting UTC to `Asia/Kuala_Lumpur`; do not add another eight hours. Remove only `timestamp`; retain all other source columns.

In [27]:
for product, product_datasets in datasets.items():
    for dataset_index, data in enumerate(product_datasets):
        source_timestamp = pd.to_datetime(
            data["timestamp"], errors="raise"
        ).dt.tz_localize("UTC")
        recorded_malaysia_datetime = pd.to_datetime(
            data["malaysia_datetime"], errors="raise"
        )
        if recorded_malaysia_datetime.dt.tz is not None:
            raise ValueError(
                f"Expected naive Malaysia-local datetime values in {product}."
            )

        malaysia_datetime = recorded_malaysia_datetime.dt.tz_localize(
            "Asia/Kuala_Lumpur"
        )
        expected_malaysia_datetime = source_timestamp.dt.tz_convert(
            "Asia/Kuala_Lumpur"
        )
        if not malaysia_datetime.equals(expected_malaysia_datetime):
            raise ValueError(
                f"Source and Malaysia timestamps do not match in "
                f"{product} dataset {dataset_index + 1}."
            )

        data["malaysia_datetime"] = malaysia_datetime
        data = (
            data.drop(columns="timestamp")
            .sort_values("malaysia_datetime", kind="stable")
            .reset_index(drop=True)
        )
        product_datasets[dataset_index] = data

        if not data["malaysia_datetime"].is_monotonic_increasing:
            raise ValueError(
                f"Timestamps are not chronological in {product} "
                f"dataset {dataset_index + 1}."
            )
        print(
            f"{product} {YEAR_GROUPS[dataset_index]}: {len(data):,} rows; "
            f"columns={data.columns.tolist()}; "
            f"timezone={data['malaysia_datetime'].dt.tz}; "
            f"range={data['malaysia_datetime'].min()} to "
            f"{data['malaysia_datetime'].max()}"
        )

CLP 2016_2020: 107,552 rows; columns=['malaysia_datetime', 'latitude', 'longitude', 'Hour', 'CLOT', 'CLTT', 'CLTH', 'CLER_23', 'CLTYPE', 'QA']; timezone=Asia/Kuala_Lumpur; range=2016-01-01 09:00:00+08:00 to 2020-12-31 18:50:00+08:00
CLP 2026: 3,594 rows; columns=['malaysia_datetime', 'latitude', 'longitude', 'Hour', 'CLOT', 'CLTT', 'CLTH', 'CLER_23', 'CLTYPE', 'QA']; timezone=Asia/Kuala_Lumpur; range=2026-06-01 09:00:00+08:00 to 2026-07-31 18:50:00+08:00
ARP 2016_2020: 107,550 rows; columns=['malaysia_datetime', 'latitude', 'longitude', 'Hour', 'AOT', 'AOT_uncertainty', 'QA']; timezone=Asia/Kuala_Lumpur; range=2016-01-01 09:00:00+08:00 to 2020-12-31 18:50:00+08:00
ARP 2026: 3,594 rows; columns=['malaysia_datetime', 'latitude', 'longitude', 'Hour', 'AOT', 'AOT_uncertainty', 'QA']; timezone=Asia/Kuala_Lumpur; range=2026-06-01 09:00:00+08:00 to 2026-07-31 18:50:00+08:00


## Task 2: Filter 2026 Datasets to 10:00~17:00

For both ARP and CLP, keep only 2026 observations whose Malaysia-local time is between 10:00:00 and 17:00:00, inclusive. The 2016?2020 datasets remain unchanged.

In [28]:
START_TIME = time(10, 0)
END_TIME = time(17, 0)

for product, product_datasets in datasets.items():
    data_2026 = product_datasets[1]
    local_times = data_2026["malaysia_datetime"].dt.time
    time_filter = local_times.between(START_TIME, END_TIME, inclusive="both")
    data_2026 = data_2026.loc[time_filter].reset_index(drop=True)
    product_datasets[1] = data_2026

    print(f"{product}_2026 rows after filtering: {len(data_2026):,}")
    print(
        "Included boundary rows: "
        f"10:00={int(data_2026['malaysia_datetime'].dt.time.eq(START_TIME).sum()):,}, "
        f"17:00={int(data_2026['malaysia_datetime'].dt.time.eq(END_TIME).sum()):,}"
    )

CLP_2026 rows after filtering: 2,558
Included boundary rows: 10:00=61, 17:00=61
ARP_2026 rows after filtering: 2,558
Included boundary rows: 10:00=61, 17:00=61


## Task 3: Change the Time Interval

Aggregate each calendar date separately into right-closed intervals labeled by the interval end. For example, `(10:00, 11:00]` is labeled `11:00`. Continuous numeric measurements use arithmetic means, while categorical `CLTYPE` and `QA` retain the last observed value when present. Each product's 2016?2020 data uses 1-hour intervals; 2026 data uses 10-minute intervals. Empty intervals are omitted, with no filling or interpolation.

In [29]:
RESAMPLE_FREQUENCIES = {"2016_2020": "1h", "2026": "10min"}
CATEGORICAL_COLUMNS = ["CLTYPE", "QA"]
resampled_datasets = {}

for product, product_datasets in datasets.items():
    resampled_product_datasets = []
    for year_group, data in zip(YEAR_GROUPS, product_datasets, strict=True):
        frequency = RESAMPLE_FREQUENCIES[year_group]
        daily_results = []
        value_columns = [column for column in data.columns if column != "malaysia_datetime"]
        mean_columns = [
            column
            for column in value_columns
            if pd.api.types.is_numeric_dtype(data[column])
            and column not in CATEGORICAL_COLUMNS
        ]
        categorical_columns = [
            column for column in CATEGORICAL_COLUMNS if column in data.columns
        ]
        for _, day_data in data.groupby(data["malaysia_datetime"].dt.date, sort=True):
            resampler = day_data.set_index(
                "malaysia_datetime", drop=False
            ).resample(
                frequency,
                closed="right",
                label="right",
                origin="start_day",
            )
            interval_data = resampler[mean_columns].mean()
            interval_data["malaysia_datetime"] = interval_data.index
            for column in categorical_columns:
                interval_data[column] = resampler[column].last()
            interval_data = interval_data.loc[resampler.size().gt(0)]
            daily_results.append(interval_data.reset_index(drop=True))

        resampled_data = pd.concat(daily_results, ignore_index=True)
        resampled_data = resampled_data.loc[:, data.columns]
        if not resampled_data["malaysia_datetime"].is_monotonic_increasing:
            raise ValueError(
                f"Resampled timestamps are not chronological in {product} {year_group}."
            )
        resampled_product_datasets.append(resampled_data)
        print(f"{product}_{year_group} ({frequency} intervals): {len(resampled_data):,} rows")

    resampled_datasets[product] = resampled_product_datasets

datasets = resampled_datasets

CLP_2016_2020 (1h intervals): 20,081 rows
CLP_2026 (10min intervals): 2,558 rows
ARP_2016_2020 (1h intervals): 20,081 rows
ARP_2026 (10min intervals): 2,558 rows


## Task 4: Exclude Unneeded Columns

Before saving the processed datasets, remove `latitude`, `longitude`, `Hour`, `CLTT`, and `QA` from CLP; remove `latitude`, `longitude`, `Hour`, `AOT_uncertainty`, and `QA` from ARP. Apply these exclusions to both year groups. The original source CSVs remain unchanged.

In [30]:
EXCLUDED_COLUMNS = {
    "CLP": ("latitude", "longitude", "Hour", "CLTT", "QA"),
    "ARP": ("latitude", "longitude", "Hour", "AOT_uncertainty", "QA"),
}

for product, product_datasets in datasets.items():
    excluded_columns = EXCLUDED_COLUMNS[product]
    for index, data in enumerate(product_datasets):
        missing_columns = set(excluded_columns) - set(data.columns)
        if missing_columns:
            raise ValueError(
                f"Cannot exclude missing {product} columns: {sorted(missing_columns)}"
            )
        product_datasets[index] = data.drop(columns=list(excluded_columns))
    print(
        f"{product}: excluded {', '.join(excluded_columns)}; "
        f"remaining columns={product_datasets[0].columns.tolist()}"
    )

CLP: excluded latitude, longitude, Hour, CLTT, QA; remaining columns=['malaysia_datetime', 'CLOT', 'CLTH', 'CLER_23', 'CLTYPE']
ARP: excluded latitude, longitude, Hour, AOT_uncertainty, QA; remaining columns=['malaysia_datetime', 'AOT']


In [31]:
PROCESSED_DIR = PROJECT_ROOT / "dataset" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

for product, product_datasets in datasets.items():
    for year_group, data in zip(YEAR_GROUPS, product_datasets, strict=True):
        filename = f"{product}_{year_group}_processed.csv"
        output_file = PROCESSED_DIR / filename
        data.to_csv(output_file, index=False)
        print(f"Saved {len(data):,} rows to {output_file}")

Saved 20,081 rows to c:\FYP\coding\dataset\processed\CLP_2016_2020_processed.csv
Saved 2,558 rows to c:\FYP\coding\dataset\processed\CLP_2026_processed.csv
Saved 20,081 rows to c:\FYP\coding\dataset\processed\ARP_2016_2020_processed.csv
Saved 2,558 rows to c:\FYP\coding\dataset\processed\ARP_2026_processed.csv
